# Welcome to the Backend!

This notebook is executed in 4 steps:
1. Installing dependencies
2. Setting up the RAG-based model
3. Setting up the servers, including the: </br>
    3a. Flask server (to connect to frontened) </br>
    3b. ngrok tunnel (to connect to Collab by forwarding traffic to external server) </br>
4. Passing the ngrok URL to the frontend!

## 1. Install Dependencies

In [1]:
!pip install flask flask-cors transformers accelerate pyngrok
!pip install -U bitsandbytes>=0.46.1
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 11.0 MB/s eta 0:00:00


For the model.

In [2]:
import gc
import torch
import sys
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer

For Flask.

In [3]:
from flask import Flask, request, jsonify, Response, stream_with_context
from flask_cors import CORS
from pyngrok import ngrok
from threading import Thread
import threading

For RAG.

In [4]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from google.colab import drive
import numpy as np
import os
import re
import pickle

## 2. Set up the RAG-Based Model

Clean the environment.

In [5]:
for name in ["model", "processor", "tokenizer"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

!rm -rf /root/.cache/huggingface/hub
!rm -rf /root/.cache/torch
!rm -rf /root/.cache/transformers
!rm -rf /root/.cache/pip

!df -h
!nvidia-smi # ensures NVIDIA drivers are installed and visible

Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   44G  193G  19% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G     0  5.7G   0% /dev/shm
/dev/root       2.0G  1.2G  748M  63% /usr/sbin/docker-init
/dev/sda1       242G   49G  194G  20% /opt/bin/.nvidia
tmpfs           6.4G   48K  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
Thu May  7 16:01:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|              

Check GPU visibility.

In [6]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
  print("GPU", torch.cuda.get_device_name(0))
  print("GPU memory allocated:", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")
  print("GPU memory reserved:", round(torch.cuda.memory_reserved(0) / 1024**3, 2), "GB")
else:
  print("No GPU found. Go to Runtime > Change runtime type > GPU")

CUDA available: True
GPU Tesla T4
GPU memory allocated: 0.0 GB
GPU memory reserved: 0.0 GB


### Integrate with RAG (PDF-based chunks)

Set up embedder.

In [7]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunks: list[str] = []           # populated by load_books()
chunk_embeddings: np.ndarray     # populated by load_books()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Load in chunks from PDF. </br>
**NOTE**: This is where we would implement spoiler-based screening.

In [8]:
CACHE_PATH = "/content/drive/MyDrive/lisan_al_gpt_cache.pkl"

def _split_sentences(text: str) -> list[str]:
    # Split at sentence-ending punctuation only when the next word is capitalized,
    # which avoids false breaks on abbreviations like "Mr." or "Dr."
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
    return [p.strip() for p in parts if p.strip()]

def _pack_chunks(sentences: list[str], chunk_size: int, overlap: int) -> list[str]:
    """Greedily pack complete sentences into word-budget chunks with sentence-level overlap."""
    chunks = []
    i = 0
    while i < len(sentences):
        word_count, j = 0, i
        while j < len(sentences):
            n = len(sentences[j].split())
            if word_count + n > chunk_size and j > i:
                break
            word_count += n
            j += 1
        chunks.append(" ".join(sentences[i:j]))
        # Step back enough sentences to cover ~overlap words for the next chunk's prefix
        overlap_words, step = 0, j
        while step > i + 1:
            step -= 1
            overlap_words += len(sentences[step].split())
            if overlap_words >= overlap:
                break
        i = step if step < j else j
    return chunks

def load_books(pdf_paths: list[str], chunk_size: int = 200, overlap: int = 50, force_reload: bool = False):
    """Index one or more PDFs into sentence-boundary chunks + embeddings. Caches to Drive."""
    global chunks, chunk_embeddings

    if not force_reload and os.path.exists(CACHE_PATH):
        print("Loading cached embeddings from Drive...")
        with open(CACHE_PATH, "rb") as f:
            data = pickle.load(f)
        chunks = data["chunks"]
        chunk_embeddings = data["chunk_embeddings"]
        print(f"Loaded {len(chunks)} cached chunks.")
        return

    print("Building chunk index from PDFs...")
    all_sentences = []
    for path in pdf_paths:
        reader = PdfReader(path)
        text = " ".join(page.extract_text() or "" for page in reader.pages)
        all_sentences.extend(_split_sentences(text))

    chunks = _pack_chunks(all_sentences, chunk_size, overlap)
    print(f"Created {len(chunks)} chunks. Encoding...")
    chunk_embeddings = embedder.encode(
        chunks, show_progress_bar=True, batch_size=64, normalize_embeddings=True
    )

    with open(CACHE_PATH, "wb") as f:
        pickle.dump({"chunks": chunks, "chunk_embeddings": chunk_embeddings}, f)
    print(f"Cached {len(chunks)} chunks to Drive.")

In [9]:
def retrieve(query: str, top_k: int = 3, min_score: float = 0.25) -> list[str]:
    """Return the top_k most relevant chunks, skipping any below min cosine similarity."""
    query_vec = embedder.encode([query], normalize_embeddings=True)
    scores = np.dot(chunk_embeddings, query_vec.T).squeeze()
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [chunks[i] for i in top_indices if scores[i] >= min_score]

Load in model.

In [10]:
SYSTEM_PROMPT = """
You are Lisan al-GPT, a Dune-focused question-answering chatbot.

For this baseline experiment, answer using only your pretrained knowledge.
Do not claim to have access to the books, retrieved passages, a database, or citations.
If you are uncertain, say so clearly.
If the user asks for no spoilers past a certain book or chapter, respect that spoiler boundary.
Keep answers concise but informative.
"""

In [11]:
print("Loading model on GPU..." if torch.cuda.is_available() else "Loading on CPU...")

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True),
    device_map="auto"
)

Loading model on GPU...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, device_map='auto')
model.eval()
# ensure model was loaded correctly
print("Loaded:", MODEL_ID)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded: Qwen/Qwen2.5-7B-Instruct


In [13]:
def generate(prompt: str, max_new_tokens: int = 512, temperature: float = 0.2,
             use_rag: bool = True, streamer=None) -> str:

    if use_rag and len(chunks) > 0:
        relevant_chunks = retrieve(prompt)
        if relevant_chunks:
            context = "\n\n---\n\n".join(relevant_chunks)
            system_content = (
                f"{SYSTEM_PROMPT}\n\n"
                f"Use the following context to help answer the user's question "
                f"if it is relevant. If it isn't relevant, answer normally.\n\n"
                f"Context:\n{context}"
            )
        else:
            system_content = SYSTEM_PROMPT
    else:
        system_content = SYSTEM_PROMPT

    messages = [
        {"role": "system", "content": system_content},
        {"role": "user",   "content": prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    device = next(model.parameters()).device
    inputs = tokenizer([text], return_tensors="pt").to(device)

    generation_kwargs = {
        **inputs,
        "max_new_tokens": max_new_tokens,
        "do_sample": temperature > 0,
    }
    if temperature > 0:
        generation_kwargs["temperature"] = temperature
    if streamer is not None:
        generation_kwargs["streamer"] = streamer

    with torch.inference_mode():
        generated_ids = model.generate(**generation_kwargs)

    if streamer is not None:
        return ""

    generated_ids_trimmed = generated_ids[:, inputs.input_ids.shape[1]:]
    return tokenizer.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]

Check to make sure the model fits on the GPU.

In [14]:
# check to make sure everything fits on GPU
devices = set(param.device for param in model.parameters());

# Compare against an actual device object
if devices == {torch.device('cuda', 0)}:
    print("PASSED: All parameters are on GPU.")
elif all(d.type == 'cuda' for d in devices):
    print("PASSED: All parameters are on GPU.")
else:
    print("FAILED: Some parameters are NOT on GPU.")
    # Show which devices are being used
    print("Devices found:", devices)
    sys.exit() # quit if not

PASSED: All parameters are on GPU.


Chunk PDFs.

In [15]:
drive.mount('/content/drive')

BOOK_NAMES = [
    "Dune.pdf",
    "Dune-Messiah.pdf",
    "Children-of-Dune.pdf",
    "God-Emperor-of-Dune.pdf",
    "Heretics-of-Dune.pdf",
    "Chapterhouse-Dune.pdf",
]

found_books = {}
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if f in BOOK_NAMES:
            found_books[f] = os.path.join(root, f)

missing = [name for name in BOOK_NAMES if name not in found_books]
if missing:
    print("Warning — missing books:", missing)

book_paths = [found_books[name] for name in BOOK_NAMES if name in found_books]
load_books(book_paths)

Mounted at /content/drive
Loading cached embeddings from Drive...
Loaded 6992 cached chunks.


## 3a. Flask Server

In [16]:
app = Flask(__name__)
CORS(app)

@app.route("/generate", methods=["POST"])
def generate_route():
    data = request.json
    prompt = data.get("prompt", "")

    streamer = TextIteratorStreamer(tokenizer, skip_special_tokens=True, skip_prompt=True)

    Thread(target=generate, kwargs={"prompt": prompt, "streamer": streamer}).start()

    def token_stream():
        for token in streamer:
            yield token

    return Response(
        stream_with_context(token_stream()),
        mimetype="text/plain",
        headers={"X-Accel-Buffering": "no", "Cache-Control": "no-cache"},
    )

# Kill any leftover process on port 5000 from a previous run, then start fresh
os.system("fuser -k 5000/tcp 2>/dev/null || true")
time.sleep(0.5)
threading.Thread(target=lambda: app.run(port=5000, use_reloader=False, threaded=True)).start()

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


## 3b. ngrok Tunnel

In [ ]:
auth_token="AHHHHHHHH"
ngrok.set_auth_token(auth_token)

# Kill any existing ngrok agent (closes all tunnels) before opening a new one
ngrok.kill()

# Open a public tunnel to port 5000
public_url = ngrok.connect(5000)
print("Your public API URL=", public_url)

Your public API URL= NgrokTunnel: "https://upwind-stylist-dwelling.ngrok-free.dev" -> "http://localhost:5000"


INFO:werkzeug:127.0.0.1 - - [07/May/2026 16:06:51] "OPTIONS /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 16:06:57] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 16:07:33] "OPTIONS /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/May/2026 16:07:37] "POST /generate HTTP/1.1" 200 -
